In [ ]:
# P0.3 (replikasi): pin stack era 4.x (transformers 5.0 menurunkan performa).
# torchao 0.10 tidak kompatibel dengan peft -> uninstall dulu.
!pip uninstall -y torchao
!pip install --force-reinstall --no-deps "transformers==4.46.3" "peft==0.13.2" "tokenizers==0.20.3" "huggingface-hub==0.26.5"


In [ ]:
import os
import sys
from pathlib import Path

def resolve_path(filename):
    """Cari file secara rekursif di /kaggle/input (Kaggle) atau kandidat lokal."""
    # 1. Rekursif cari di /kaggle/input (menangani semua variasi mount Kaggle)
    if Path("/kaggle/input").exists():
        for root, _dirs, files in os.walk("/kaggle/input"):
            if filename in files:
                found = os.path.join(root, filename)
                print(f"[resolve_path] Ditemukan di Kaggle: {found}")
                return found
    # 2. Kandidat lokal workstation
    candidates = [
        Path(f"Data/processed/{filename}"),
        Path(f"Data/simulated/{filename}"),
        Path(f"Data/raw/{filename}"),
        Path(f"Data/{filename}"),
        Path(f"kamus/{filename}"),
        Path(f"Output/predictions/{filename}"),
        Path(f"../Data/processed/{filename}"),
        Path(f"../Data/simulated/{filename}"),
        Path(f"../Data/raw/{filename}"),
        Path(f"../kamus/{filename}"),
        Path(filename),
    ]
    for p in candidates:
        if p.exists():
            print(f"[resolve_path] Ditemukan lokal: {p}")
            return str(p)
    return filename


In [ ]:
# P0.1 (replikasi): paksa 1 GPU (DataParallel menggandakan batch -> undertrained).
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))

import sys
import torch
import transformers
import peft

assert transformers.__version__.startswith("4.46"), (
    f"transformers {transformers.__version__} bukan pin 4.46 - instalasi bermasalah!"
)
from transformers import TFPreTrainedModel  # bukti tidak ada file campur 5.0

print("python        :", sys.version)
print("torch         :", torch.__version__)
print("transformers  :", transformers.__version__)
print("peft          :", peft.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count     :", torch.cuda.device_count())


In [ ]:
from __future__ import annotations
# =====================================================
# SUMBER KEBENARAN: src/ (disuntik oleh tools/generate_notebook.py)
# Jangan edit langsung di notebook - edit src/ lalu generate ulang.
# =====================================================

# --- src/config.py ---
"""Konfigurasi eksperimen: load dari YAML/JSON dan bantu membenamkan dict ke notebook.

Sumber kebenaran konfigurasi = file di `configs/`. Generator membaca file ini,
lalu membenamkan representasi literal dict-nya ke sel Config notebook (sel 6),
sehingga notebook Kaggle tidak butuh PyYAML.
"""

import json
from pathlib import Path
from typing import Any


def load_config(path: str | Path) -> dict[str, Any]:
    """Muat file config (.yaml/.yml/.json) menjadi dict."""
    p = Path(path)
    text = p.read_text(encoding="utf-8").strip()
    if p.suffix.lower() in (".yaml", ".yml"):
        try:
            import yaml
        except ImportError as e:
            raise ImportError(
                "PyYAML dibutuhkan untuk config YAML: pip install pyyaml"
            ) from e
        cfg = yaml.safe_load(text)
        if not isinstance(cfg, dict):
            raise ValueError(f"Config {p} harus berupa mapping YAML, bukan {type(cfg)}")
        return cfg
    return json.loads(text)


import pprint


def config_repr(cfg: dict[str, Any]) -> str:
    """Representasi Python literal (pprint.pformat) untuk dibenamkan di sel notebook."""
    return pprint.pformat(cfg, indent=4, sort_dicts=False)


def config_snippet(cfg: dict[str, Any], var_name: str = "CONFIG") -> str:
    """Source untuk sel Config: `CONFIG = {...}`."""
    return f"{var_name} = {config_repr(cfg)}"

# --- src/data.py ---
"""Data: loader dataset fleksibel (path mount Kaggle CLI 2.x vs lama), split, dataset PyTorch.

Sumber kebenaran loading data untuk semua eksperimen. Sel notebook menyuntik source
fungsi-fungsi di sini (via generator) sehingga notebook tetap self-contained di Kaggle.
"""

import os
from typing import Any

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset

COL_TEXT = "text_bert"
COL_LABEL = "label"
CSV_NAME = "data_preprocessed_with_emoticon.csv"


def find_dataset_csv(csv_name: str = CSV_NAME) -> str:
    """Cari CSV dataset di /kaggle/input (path mount berubah antara CLI 2.x dan lama).

    - CLI 2.x:  /kaggle/input/datasets/<owner>/<slug>/...
    - Skema lama: /kaggle/input/<slug>/...
    Tidak ada hardcode path: cari dari daftar file ter-mount.
    """
    mounted = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f == csv_name:
                mounted.append(os.path.join(root, f))
    if not mounted:
        raise FileNotFoundError(
            f"Dataset '{csv_name}' tidak ditemukan di /kaggle/input. "
            "Cek dataset_sources di kernel-metadata.json."
        )
    return mounted[0]


def load_dataframe(
    csv_name: str = CSV_NAME,
    col_text: str = COL_TEXT,
    col_label: str = COL_LABEL,
) -> pd.DataFrame:
    """Muat CSV dataset dengan validasi kolom teks eksplisit."""
    path = find_dataset_csv(csv_name)
    print("CSV ditemukan di:", path)
    df = pd.read_csv(path)
    if col_text not in df.columns:
        raise ValueError(
            f"Kolom '{col_text}' tidak ditemukan di CSV. Kolom tersedia: {df.columns.tolist()}"
        )
    df[col_text] = df[col_text].fillna("").astype(str)
    print(f"Kolom BERT terpilih: {col_text} | Total baris: {len(df)}")
    return df


def split_data(
    df: pd.DataFrame,
    test_size: float = 0.2,
    val_size: float = 0.1,
    random_state: int = 42,
    col_text: str | None = None,
    col_label: str | None = None,
) -> dict[str, np.ndarray]:
    """Split 80:20 (test) lalu 90:10 (val) — protokol konsisten semua eksperimen."""
    ct = col_text or COL_TEXT
    cl = col_label or COL_LABEL
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df[cl]
    )
    X_train = train_df[ct].values
    X_test = test_df[ct].values
    y_train = train_df[cl].values
    y_test = test_df[cl].values

    X_train_final, X_val, y_train_final, y_val = train_test_split(
        X_train, y_train, test_size=val_size, stratify=y_train, random_state=random_state
    )
    return {
        "X_train": X_train_final,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train_final,
        "y_val": y_val,
        "y_test": y_test,
    }


class SentimenDataset(Dataset):
    """Dataset PyTorch numpy-friendly (menerima numpy array & pandas Series)."""

    def __init__(
        self,
        texts: Any,
        labels: Any,
        tokenizer,
        max_length: int = 128,
    ):
        self.texts = texts.values if isinstance(texts, pd.Series) else np.array(texts)
        self.labels = labels.values if isinstance(labels, pd.Series) else np.array(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }


class EncodedDataset(Dataset):
    """Dataset PyTorch dari encodings pre-tokenized (input_ids + attention_mask + labels)."""

    def __init__(self, encodings: dict[str, torch.Tensor], labels: Any):
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        self.labels = labels.values if isinstance(labels, pd.Series) else np.array(labels)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }


def build_simulated_scenario(
    texts: Any,
    labels: Any,
    targets: dict[int, int],
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """Bentuk skenario ketimpangan kelas deterministik HANYA dari partisi train (zero leakage).

    Replikasi persis protokol experiments/generate_simulated_data.py:
    sample n sampel per kelas tanpa pengembalian (random_state=seed) lalu shuffle penuh.
    """
    df = pd.DataFrame({"text": np.array(texts), "label": np.array(labels)})
    dfs = []
    for label_val, target_n in targets.items():
        class_subset = df[df["label"] == label_val]
        if len(class_subset) < target_n:
            raise ValueError(
                f"Insufficient samples for class {label_val}: available {len(class_subset)}, requested {target_n}"
            )
        dfs.append(class_subset.sample(n=target_n, replace=False, random_state=seed))
    scenario_df = pd.concat(dfs, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return scenario_df["text"].values, scenario_df["label"].values.astype(np.int64)


def apply_balancing(
    texts: Any,
    labels: Any,
    strategy: str,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, list[float] | None]:
    """Terapkan strategi penyeimbangan pada teks latih (mirror protokol M8).

    Returns: (texts_balanced, labels_balanced, class_weight_list_or_None).
    """
    texts_arr = np.array(texts)
    labels_arr = np.array(labels)

    if strategy == "baseline":
        return texts_arr, labels_arr, None

    if strategy == "class_weight":
        classes = np.array([0, 1, 2])
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels_arr)
        return texts_arr, labels_arr, [float(w) for w in weights]

    if strategy == "ros":
        unique, counts = np.unique(labels_arr, return_counts=True)
        max_c = max(counts)
        dfs_x, dfs_y = [], []
        rng = np.random.default_rng(seed)
        for cls_val in unique:
            idx = np.where(labels_arr == cls_val)[0]
            if len(idx) < max_c:
                resampled_idx = rng.choice(idx, size=max_c, replace=True)
                dfs_x.append(texts_arr[resampled_idx])
                dfs_y.append(labels_arr[resampled_idx])
            else:
                dfs_x.append(texts_arr[idx])
                dfs_y.append(labels_arr[idx])
        X_bal = np.concatenate(dfs_x, axis=0)
        y_bal = np.concatenate(dfs_y, axis=0)
        perm = rng.permutation(len(y_bal))
        return X_bal[perm], y_bal[perm], None

    if strategy == "rus":
        unique, counts = np.unique(labels_arr, return_counts=True)
        min_c = min(counts)
        dfs_x, dfs_y = [], []
        rng = np.random.default_rng(seed)
        for cls_val in unique:
            idx = np.where(labels_arr == cls_val)[0]
            if len(idx) > min_c:
                resampled_idx = rng.choice(idx, size=min_c, replace=False)
                dfs_x.append(texts_arr[resampled_idx])
                dfs_y.append(labels_arr[resampled_idx])
            else:
                dfs_x.append(texts_arr[idx])
                dfs_y.append(labels_arr[idx])
        X_bal = np.concatenate(dfs_x, axis=0)
        y_bal = np.concatenate(dfs_y, axis=0)
        perm = rng.permutation(len(y_bal))
        return X_bal[perm], y_bal[perm], None

    raise ValueError(f"Unknown strategy: {strategy} (pilihan: baseline, class_weight, ros, rus)")


class MLMDataset(Dataset):
    """Dataset PyTorch untuk Masked Language Modeling (TAPT)."""

    def __init__(self, texts: Any, tokenizer, max_length: int = 128):
        self.texts = texts.values if isinstance(texts, pd.Series) else np.array(texts)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }

# --- src/model.py ---
"""Model: builder IndoBERTweet-LoRA (sumber kebenaran tunggal)."""

from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    PreTrainedTokenizerFast,
)

MODEL_NAME = "indolem/indobertweet-base-uncased"
ID2LABEL = {0: "negatif", 1: "netral", 2: "positif"}
LABEL2ID = {"negatif": 0, "netral": 1, "positif": 2}


def build_indobertweet_lora(
    dropout: float = 0.3,
    r: int = 16,
    lora_alpha: int = 32,
    num_labels: int = 3,
    pretrained_model_name_or_path: str = MODEL_NAME,
):
    """Bangun model IndoBERTweet + LoRA (r, alpha, dropout) dengan classifier baru."""
    config = AutoConfig.from_pretrained(
        pretrained_model_name_or_path,
        num_labels=num_labels,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path, config=config, ignore_mismatched_sizes=True
    )
    model.config.id2label = ID2LABEL
    model.config.label2id = LABEL2ID

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["query", "value"],
        lora_dropout=dropout,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        modules_to_save=["classifier"],
    )
    return get_peft_model(model, lora_config)


def load_tokenizer() -> PreTrainedTokenizerFast:
    from transformers import AutoTokenizer

    return AutoTokenizer.from_pretrained(MODEL_NAME)

# --- src/metrics.py ---
"""Metrik evaluasi: compute_metrics (HF) + softmax numpy (untuk simpan probabilitas)."""

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

LABEL_NAMES = ["negatif", "netral", "positif"]


def compute_metrics(eval_pred):
    """Metrik untuk HF Trainer (average='macro', zero_division=0)."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


def softmax_np(logits: np.ndarray) -> np.ndarray:
    """Softmax stabil (numerik) di axis=1."""
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def prediction_frame(
    texts,
    y_true,
    logits: np.ndarray,
    prob_cols=("prob_negatif", "prob_netral", "prob_positif"),
):
    """DataFrame per-sampel: teks, label aktual/prediksi, probabilitas per kelas."""
    import pandas as pd

    y_pred = np.argmax(logits, axis=1)
    P = softmax_np(logits)
    return pd.DataFrame(
        {
            "text": pd.Series(texts),
            "label_aktual": pd.Series(y_true),
            "label_prediksi": pd.Series(y_pred),
            prob_cols[0]: P[:, 0],
            prob_cols[1]: P[:, 1],
            prob_cols[2]: P[:, 2],
        }
    )

# --- src/trainer_factory.py ---
"""Trainer Factory: satu fungsi untuk membangun Trainer dengan loss yang dipilih.

Sumber kebenaran tunggal untuk:
- "cross_entropy" -> Trainer standar HF
- "weighted_ce"   -> WeightedTrainer (CrossEntropyLoss(weight=class_weight))
- "focal"         -> FocalLossTrainer (gamma, alpha=class_weight)

Sel notebook menyuntik source file ini, sehingga tidak ada duplikasi kode Trainer
di antar-notebook (akar bug 'trainer_best' & compute_loss ganda di masa lalu).
"""

import torch
import torch.nn.functional as F
from torch import nn
from transformers import Trainer


class WeightedTrainer(Trainer):
    """CrossEntropyLoss dengan bobot kelas."""

    def __init__(self, class_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weight = (
            torch.tensor(class_weight, dtype=torch.float) if class_weight is not None else None
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weight is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weight.to(logits.device))
        else:
            loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class FocalLossTrainer(Trainer):
    """Focal Loss: FL(p_t) = -alpha_t (1-p_t)^gamma log(p_t), dengan alpha opsional."""

    def __init__(self, gamma=2.0, class_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.gamma = gamma
        self.class_weight = (
            torch.tensor(class_weight, dtype=torch.float) if class_weight is not None else None
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        probs = torch.softmax(logits, dim=-1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
        focal = -(1.0 - pt) ** self.gamma * torch.log(pt.clamp(min=1e-8))
        if self.class_weight is not None:
            alpha_t = self.class_weight.to(logits.device).gather(0, labels)
            focal = focal * alpha_t
        loss = focal.mean()
        return (loss, outputs) if return_outputs else loss


def build_trainer(
    loss: str = "cross_entropy",
    class_weight=None,
    gamma: float = 2.0,
    **trainer_kwargs,
) -> Trainer:
    """Bangun Trainer sesuai strategi loss.

    Contoh:
        build_trainer(loss="weighted_ce", class_weight=[0.75, 1.32, 1.03], args=args, ...)
        build_trainer(loss="focal", gamma=2.0, class_weight=[...], args=args, ...)
    """
    if loss == "cross_entropy":
        trainer_cls = Trainer
    elif loss == "weighted_ce":
        trainer_cls = WeightedTrainer
    elif loss == "focal":
        trainer_cls = FocalLossTrainer
    else:
        raise ValueError(f"Loss tidak dikenal: {loss} (pilihan: cross_entropy, weighted_ce, focal)")

    if loss == "weighted_ce":
        return WeightedTrainer(class_weight=class_weight, **trainer_kwargs)
    if loss == "focal":
        return FocalLossTrainer(gamma=gamma, class_weight=class_weight, **trainer_kwargs)
    return Trainer(**trainer_kwargs)

# --- src/summary.py ---
"""Auto Experiment Summary: menulis metadata run (config, commit, dataset MD5) ke file & stdout."""

import hashlib
import json
import subprocess
from pathlib import Path
from typing import Any


def git_commit_short() -> str:
    """Hash commit git saat ini (atau 'unknown' bila bukan repo git)."""
    try:
        out = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        return out.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def file_md5(path: str | Path) -> str:
    """MD5 file (untuk audit data lineage)."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def experiment_summary(
    exp_id: str,
    config: dict[str, Any],
    metrics: dict[str, Any],
    csv_path: str | None = None,
    dataset_path: str | None = None,
    out_path: str | None = None,
) -> dict[str, Any]:
    """Buat dict ringkasan + tulis file JSON (opsional) + cetak ke stdout.

    Metrics contoh: {"accuracy": ..., "f1_macro": ..., "netral_f1": ...}
    """
    summary = {
        "experiment": exp_id,
        "commit": git_commit_short(),
        "config": config,
        "metrics": metrics,
    }
    if csv_path:
        summary["prediction_csv"] = csv_path
    if dataset_path:
        summary["dataset_md5"] = file_md5(dataset_path)

    if out_path:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        Path(out_path).write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    print("=" * 60)
    print("AUTO EXPERIMENT SUMMARY")
    print("=" * 60)
    print(f"Experiment : {exp_id}")
    print(f"Commit     : {summary.get('commit')}")
    if dataset_path:
        print(f"Dataset MD5: {summary['dataset_md5']}")
    print(f"Config     : {json.dumps(config, ensure_ascii=False)}")
    print(f"Metrics    : {json.dumps(metrics, ensure_ascii=False)}")
    if out_path:
        print(f"Summary    : {out_path}")
    return summary

In [ ]:
# =====================================================
# CONFIG (dibenamkan dari configs/exp_indobert_v2.yaml)
# =====================================================
CONFIG = {   'exp_id': 'exp_indobert_v2',
    'family': 'hf_lora_v2_suite',
    'title': 'Thesis IndoBERT v2',
    'description': 'Suite lengkap IndoBERTweet-LoRA pada Data Baru V2 '
                   '(banjir_processed_v2.csv / processed_text_v2) dengan '
                   'konfigurasi optimal B2 (lr 2e-4, r=8, a=16, d=0.05). 4 '
                   'varian empiris (Baseline, Class Weight, ROS, RUS) + 5 '
                   'simulasi ketimpangan (1:1:1, 6:3:1, 6:3:1+ROS, 8:1:1, '
                   '8:1:1+ROS), masing-masing 3 seed (42/123/456).\n',
    'dataset_csv': 'banjir_processed_v2.csv',
    'text_col': 'processed_text_v2',
    'label_col': 'label',
    'params': {   'learning_rate': 0.0002,
                  'epochs': 5,
                  'batch_size': 16,
                  'warmup_ratio': 0.1,
                  'weight_decay': 0.01,
                  'max_length': 128,
                  'dropout': 0.05,
                  'lora_r': 8,
                  'lora_alpha': 16,
                  'patience': 2},
    'seeds': [42, 123, 456],
    'empirical_variants': ['baseline', 'class_weight', 'ros', 'rus'],
    'simulations': [   {   'id': 'scenario_111',
                           'ratio': '1:1:1',
                           'targets': {0: 1000, 1: 1000, 2: 1000}},
                       {   'id': 'scenario_631',
                           'ratio': '6:3:1',
                           'targets': {0: 3000, 2: 1500, 1: 500}},
                       {   'id': 'scenario_811',
                           'ratio': '8:1:1',
                           'targets': {0: 3200, 1: 400, 2: 400}}],
    'sim_strategies': ['baseline', 'ros'],
    'dataset_sources': ['emanuelembuaijdak/thesis-indobert-processed-data']}
print(json.dumps(CONFIG, indent=2))

In [ ]:
# =====================================================
# SET SEED
# =====================================================
import random
import numpy as np
import torch
from transformers import set_seed

seed = 42
set_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print("GPU tersedia:", torch.cuda.is_available())


In [ ]:
# =====================================================
# DATASET (banjir_processed_v2.csv + processed_text_v2)
# =====================================================
import pandas as pd

CSV_V2 = CONFIG.get("dataset_csv", "banjir_processed_v2.csv")
COL_V2 = CONFIG.get("text_col", "processed_text_v2")
COL_LABEL_V2 = CONFIG.get("label_col", "label")

df = load_dataframe(csv_name=CSV_V2, col_text=COL_V2, col_label=COL_LABEL_V2)
split = split_data(df, test_size=0.2, val_size=0.1, random_state=42, col_text=COL_V2, col_label=COL_LABEL_V2)

max_len = CONFIG.get("params", {}).get("max_length", 128)
tokenizer = load_tokenizer()


def make_encoded_ds(texts, labels):
    enc = tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt",
    )
    return EncodedDataset(enc, labels)


train_dataset = make_encoded_ds(split["X_train"], split["y_train"])
val_dataset = make_encoded_ds(split["X_val"], split["y_val"])
test_dataset = make_encoded_ds(split["X_test"], split["y_test"])
print(f"Train {len(train_dataset)} | Val {len(val_dataset)} | Test {len(test_dataset)}")
print("Distribusi train:", pd.Series(split['y_train']).value_counts().sort_index().to_dict())
print("Distribusi val  :", pd.Series(split['y_val']).value_counts().sort_index().to_dict())
print("Distribusi test :", pd.Series(split['y_test']).value_counts().sort_index().to_dict())


In [ ]:
# =====================================================
# SUITE: 4 VARIAN EMPIRIS + 5 SIMULASI x 3 SEEDS (27 RUNS)
# =====================================================
import gc
import os
import shutil
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import EarlyStoppingCallback, TrainingArguments, set_seed

p = CONFIG["params"]
LR = p["learning_rate"]
EPOCHS = p["epochs"]
BS = p["batch_size"]
WARMUP = p["warmup_ratio"]
WD = p["weight_decay"]
PATIENCE = p.get("patience", 2)
SEEDS = CONFIG["seeds"]
EMPIRICAL_VARIANTS = CONFIG["empirical_variants"]
SIMULATIONS = CONFIG["simulations"]
SIM_STRATEGIES = CONFIG["sim_strategies"]

results_rows = []


def run_suite_trial(part, run_id, texts_tr, y_tr, strategy, scenario_id, seed):
    set_seed(seed)
    torch.cuda.empty_cache()

    texts_b, y_b, cw = apply_balancing(texts_tr, y_tr, strategy, seed)
    tr_ds = make_encoded_ds(texts_b, y_b)
    print("\n" + "=" * 70)
    print(f"RUN: {run_id} | strategy={strategy} | seed={seed}")
    print(f"Train n={len(y_b)} | distribusi: {pd.Series(y_b).value_counts().sort_index().to_dict()}")
    if cw is not None:
        print(f"Class weights: {cw}")
    print("=" * 70)

    model = build_indobertweet_lora(
        dropout=p["dropout"],
        r=p["lora_r"],
        lora_alpha=p["lora_alpha"],
    )

    out_dir = f"./results_{run_id}"
    training_args = TrainingArguments(
        output_dir=out_dir,
        learning_rate=LR,
        per_device_train_batch_size=BS,
        per_device_eval_batch_size=BS,
        num_train_epochs=EPOCHS,
        warmup_ratio=WARMUP,
        weight_decay=WD,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        seed=seed,
    )

    trainer = build_trainer(
        loss="weighted_ce" if strategy == "class_weight" else "cross_entropy",
        class_weight=cw,
        model=model,
        args=training_args,
        train_dataset=tr_ds,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )

    t0 = time.time()
    trainer.train()
    runtime_sec = round(time.time() - t0, 2)
    eval_result = trainer.evaluate()

    preds_test = trainer.predict(test_dataset)
    logits = preds_test.predictions
    y_pred = np.argmax(logits, axis=1)

    y_true = split["y_test"]
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    accuracy = accuracy_score(y_true, y_pred)
    _, recall_netral, f1_netral, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average="macro", zero_division=0
    )

    maj = pd.Series(split["y_val"]).mode()[0]
    p_maj = float((split["y_val"] == maj).mean())
    f1_maj = (2 * p_maj / (1 + p_maj)) / 3
    val_f1 = eval_result["eval_f1_macro"]
    status = "COLLAPSE" if val_f1 <= f1_maj + 1e-6 else "OK"

    fname = f"pred_{run_id}.csv"
    prediction_frame(split["X_test"], y_true, logits).to_csv(fname, index=False)

    if os.path.exists(out_dir):
        shutil.rmtree(out_dir, ignore_errors=True)
    del model, trainer, tr_ds
    torch.cuda.empty_cache()
    gc.collect()

    row = {
        "part": part,
        "scenario": scenario_id,
        "strategy": strategy,
        "seed": seed,
        "train_n": int(len(y_b)),
        "val_accuracy": round(float(eval_result["eval_accuracy"]), 4),
        "val_macro_f1": round(float(val_f1), 4),
        "test_accuracy": round(float(accuracy), 4),
        "test_macro_f1": round(float(f1_macro), 4),
        "test_precision_macro": round(float(precision_macro), 4),
        "test_recall_macro": round(float(recall_macro), 4),
        "recall_netral": round(float(recall_netral), 4),
        "f1_netral": round(float(f1_netral), 4),
        "status": status,
        "runtime_sec": runtime_sec,
    }
    results_rows.append(row)
    print(f"  -> Test Acc={accuracy*100:.2f}% | Macro F1={f1_macro*100:.2f}% | "
          f"Recall Netral={recall_netral*100:.2f}% | {status} | {runtime_sec}s")
    return row


# ========== PART A: VARIAN EMPIRIS ==========
print("\n" + "#" * 75)
print("# PART A: VARIAN EMPIRIS (Baseline, Class Weight, ROS, RUS)")
print("#" * 75)
for variant in EMPIRICAL_VARIANTS:
    for seed in SEEDS:
        run_id = f"emp_{variant}_s{seed}"
        run_suite_trial("empiris", run_id, split["X_train"], split["y_train"], variant, "empiris", seed)

emp_df = pd.DataFrame([r for r in results_rows if r["part"] == "empiris"])
emp_df.to_csv("exp_indobert_v2_empiris_results.csv", index=False)
print("\n=== HASIL EMPIRIS (TEST) ===")
print(emp_df.sort_values("test_macro_f1", ascending=False).to_string(index=False))

# ========== PART B: SKENARIO SIMULASI KETIMPANGAN ==========
print("\n" + "#" * 75)
print("# PART B: SIMULASI KETIMPANGAN (1:1:1, 6:3:1, 8:1:1) [+ROS pada 6:3:1 & 8:1:1]")
print("#" * 75)
for sim in SIMULATIONS:
    sc_id = sim["id"]
    X_sc, y_sc = build_simulated_scenario(split["X_train"], split["y_train"], sim["targets"], seed=42)
    print(f"\nSkenario {sc_id} ({sim['ratio']}): n={len(y_sc)} | "
          f"distribusi: {pd.Series(y_sc).value_counts().sort_index().to_dict()}")
    for strat in SIM_STRATEGIES:
        for seed in SEEDS:
            run_id = f"sim_{sc_id}_{strat}_s{seed}"
            run_suite_trial("simulasi", run_id, X_sc, y_sc, strat, sc_id, seed)

sim_df = pd.DataFrame([r for r in results_rows if r["part"] == "simulasi"])
sim_df.to_csv("exp_indobert_v2_simulasi_results.csv", index=False)
print("\n=== HASIL SIMULASI (TEST) ===")
print(sim_df.sort_values(["scenario", "strategy", "test_macro_f1"], ascending=[True, True, False]).to_string(index=False))

# ========== MASTER + AGREGASI MEAN+-STD ==========
master_df = pd.DataFrame(results_rows)
master_df.to_csv("exp_indobert_v2_suite_results.csv", index=False)
print("\n=== MASTER SUITE RESULTS (27 RUNS) ===")
print(master_df.to_string(index=False))

agg = (
    master_df.groupby(["part", "scenario", "strategy"])
    .agg(
        n_seeds=("seed", "count"),
        accuracy_mean=("test_accuracy", "mean"),
        accuracy_std=("test_accuracy", lambda s: s.std(ddof=1)),
        macro_f1_mean=("test_macro_f1", "mean"),
        macro_f1_std=("test_macro_f1", lambda s: s.std(ddof=1)),
        recall_netral_mean=("recall_netral", "mean"),
        f1_netral_mean=("f1_netral", "mean"),
    )
    .reset_index()
)
agg.to_csv("exp_indobert_v2_suite_summary.csv", index=False)
print("\n=== RANGKUMAN MEAN +- STD (3 SEEDS) ===")
print(agg.to_string(index=False))


In [ ]:
# =====================================================
# SAVE ARTIFACT + AUTO EXPERIMENT SUMMARY
# =====================================================
base_agg = agg[(agg["part"] == "empiris") & (agg["strategy"] == "baseline")]
metrics = {
    "total_runs": int(len(master_df)),
    "empiris_baseline_accuracy_mean": float(base_agg["accuracy_mean"].iloc[0]) if len(base_agg) else None,
    "empiris_baseline_macro_f1_mean": float(base_agg["macro_f1_mean"].iloc[0]) if len(base_agg) else None,
    "empiris_baseline_recall_netral_mean": float(base_agg["recall_netral_mean"].iloc[0]) if len(base_agg) else None,
}
summary = experiment_summary(
    exp_id="exp_indobert_v2",
    config=CONFIG,
    metrics=metrics,
    csv_path="exp_indobert_v2_suite_results.csv",
    out_path="exp_indobert_v2_suite_summary.json",
)
